# Preprocessing pipeline

## The order!
DataLoader -> stratified_group_split -> FeatureEncoder -> FeatureConstructor -> Imputer -> Scaler

In [ ]:
#imports 📦
import sys
sys.path.append('/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope')

from preprocessing import (DataLoader,
                           FeatureConstructor,
                           FeatureEncoder,
                           stratified_group_split,
                           Scaler,
                           Imputer)

#imports
import numpy as np
import pandas as pd
import os
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## Data loader 🚛

Demographic data
  - Patient number
  - Age
  - Sex
  - Adult BMI (kg/m2)
  - Child Weight (kg)
  - Child Height (cm)

Patient data

Audio text files

In [2]:
# define paths
audio_path = '/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
demographic_path = '/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/demographic_info.txt'
diagnosis_path = '/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv'

# instantiate and run
loader = DataLoader(audio_path, demographic_path, diagnosis_path)
data = loader.fit_transform(None)

#check
print(data.shape)
data.head()

(6898, 13)


,start,end,crackles,weezels,pid,chest_location,filename,disease,age,sex,adult_bmi,child_weight,child_height
0,0.022,0.364,0,0,148,Al,148_1b1_Al_sc_Meditron,URTI,4.0,M,NaN,33.0,110.0
1,0.364,2.436,0,0,148,Al,148_1b1_Al_sc_Meditron,URTI,4.0,M,NaN,33.0,110.0
2,2.436,4.636,0,0,148,Al,148_1b1_Al_sc_Meditron,URTI,4.0,M,NaN,33.0,110.0
3,4.636,6.793,0,0,148,Al,148_1b1_Al_sc_Meditron,URTI,4.0,M,NaN,33.0,110.0
4,6.793,8.750,0,0,148,Al,148_1b1_Al_sc_Meditron,URTI,4.0,M,NaN,33.0,110.0


## Stratified group split 🪚

In [3]:
#train test split for stratified groups to ensure class balance
train_data, test_data = stratified_group_split(data)

## Feature Encoder 🕵️

In [4]:
# instantiate and run
encoder = FeatureEncoder()
data = encoder.fit_transform(data)

#check
print(data.shape)
data.head()

(6898, 26)


,start,end,crackles,weezels,pid,filename,age,sex,adult_bmi,child_weight,...,disease_LRTI,disease_Pneumonia,disease_URTI,chest_location_Al,chest_location_Ar,chest_location_Ll,chest_location_Lr,chest_location_Pl,chest_location_Pr,chest_location_Tc
0,0.022,0.364,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,NaN,33.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.364,2.436,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,NaN,33.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2.436,4.636,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,NaN,33.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4.636,6.793,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,NaN,33.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,6.793,8.750,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,NaN,33.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


## Feature Constructor 👷

In [5]:
# instantiate and run
constructor = FeatureConstructor()
data = constructor.fit_transform(data)

#check
print(data.shape)
data.head()

(6898, 23)


,crackles,weezels,pid,filename,age,sex,disease_Asthma,disease_Bronchiectasis,disease_Bronchiolitis,disease_COPD,...,disease_URTI,chest_location_Al,chest_location_Ar,chest_location_Ll,chest_location_Lr,chest_location_Pl,chest_location_Pr,chest_location_Tc,cycle_length,bmi
0,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.342,1.704545
1,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.072,1.704545
2,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.200,1.704545
3,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.157,1.704545
4,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.957,1.704545


## Imputer ✨

In [6]:
# instantiate and run
imputer = Imputer()
data = imputer.fit_transform(data)

#check
print(data.shape)
data.head()

(6898, 23)


,crackles,weezels,pid,filename,age,sex,disease_Asthma,disease_Bronchiectasis,disease_Bronchiolitis,disease_COPD,...,disease_URTI,chest_location_Al,chest_location_Ar,chest_location_Ll,chest_location_Lr,chest_location_Pl,chest_location_Pr,chest_location_Tc,cycle_length,bmi
0,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.342,1.704545
1,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.072,1.704545
2,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.200,1.704545
3,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.157,1.704545
4,0,0,148,148_1b1_Al_sc_Meditron,4.0,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.957,1.704545


## Scaler 📐

In [7]:
# instantiate and run
scaler = Scaler()
data = scaler.fit_transform(data)

#check
print(data.shape)
data.head()

(6898, 23)


,crackles,weezels,pid,filename,age,sex,disease_Asthma,disease_Bronchiectasis,disease_Bronchiolitis,disease_COPD,...,disease_URTI,chest_location_Al,chest_location_Ar,chest_location_Ll,chest_location_Lr,chest_location_Pl,chest_location_Pr,chest_location_Tc,cycle_length,bmi
0,0,0,148,148_1b1_Al_sc_Meditron,-2.545206,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.011609,2.999509
1,0,0,148,148_1b1_Al_sc_Meditron,-2.545206,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.536065,2.999509
2,0,0,148,148_1b1_Al_sc_Meditron,-2.545206,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.426892,2.999509
3,0,0,148,148_1b1_Al_sc_Meditron,-2.545206,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.463567,2.999509
4,0,0,148,148_1b1_Al_sc_Meditron,-2.545206,1,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.634150,2.999509
